# Experiment 2c — Rescore exp2b generations at all probe positions

Loads the saved exp2b `result.json` (already has every prompt + generated response),
rebuilds `[prompt + response]` for each row, runs **one extra forward pass per row**
with hooks at every layer, captures activations at `response_first`, `response_mid`,
`response_last`, and projects each onto the matching exp1b probe direction.

No regeneration — purely rescoring. Optionally joins judge verdicts from a
judged_*.csv to give you per-position correlations against compliance.

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!pip install -q -e {REPO_DIR}
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()
import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"            # must match the model that generated the exp2b responses
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

slug = MODEL_CONFIGS[MODEL_KEY].slug
EXP1_DIR = DRIVE_ROOT / slug / "exp1b_authority_conflict"
EXP2B_RESULT_JSON = DRIVE_ROOT / slug / "exp2b_conflict_evolved" / "result.json"
JUDGE_CSV = DRIVE_ROOT / slug / "exp2b_conflict_evolved" / "judged_nothink.csv"  # optional

OUT_CSV = DRIVE_ROOT / slug / "exp2b_conflict_evolved" / "rescored_all_positions.csv"

for p in (EXP1_DIR, EXP2B_RESULT_JSON, JUDGE_CSV):
    print(f"{p}  exists={p.exists()}")
print(f"OUT_CSV = {OUT_CSV}")

In [ ]:
# Run rescoring
from mech_spoof.experiments import rescore_exp2b_at_all_positions

summary = rescore_exp2b_at_all_positions(
    model_key=MODEL_KEY,
    exp1_dir=EXP1_DIR,
    exp2b_result_path=EXP2B_RESULT_JSON,
    out_csv_path=OUT_CSV,
    judge_csv_path=JUDGE_CSV if JUDGE_CSV.exists() else None,
    batch_size=4,             # bump on A100 80GB; halve on T4
    max_length=4096,
    max_response_chars=4000,
)
summary

## Per-position correlations

In [ ]:
# Per-position correlation table at the best layer
import pandas as pd

corrs = summary.get("correlations_at_best_layer", {})
if not corrs:
    print("No judge CSV provided — correlations skipped.")
else:
    rows = []
    for pos, e in corrs.items():
        rows.append({
            "position": pos,
            "n": e.get("n"),
            "r": e.get("r"),
            "p": e.get("p"),
        })
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Per-layer correlation curves for each position (uses the rescored CSV directly)
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

df = pd.read_csv(OUT_CSV)
if "system_followed" in df.columns:
    s = df["system_followed"].astype(str).str.lower()
    df["y"] = s.where(s.isin(["true","false"])).map({"true":1,"false":0})
else:
    df["y"] = np.nan

positions = ("response_first", "response_mid", "response_last")
fig, ax = plt.subplots(figsize=(11, 5))

if df["y"].notna().any():
    for pos in positions:
        col = f"probe_scores_{pos}_by_layer_json"
        if col not in df.columns:
            continue
        layer_to_scores = {}
        for _, r in df.iterrows():
            if not isinstance(r[col], str) or not r[col].strip():
                continue
            sbl = json.loads(r[col])
            for L, sc in sbl.items():
                layer_to_scores.setdefault(int(L), []).append((float(sc), r["y"]))
        layers, rs = [], []
        for L in sorted(layer_to_scores):
            arr = layer_to_scores[L]
            arr = [(x, y) for x, y in arr if y == y]
            if len(arr) < 3:
                continue
            xs, ys = zip(*arr)
            if len(set(ys)) < 2:
                continue
            r_val, _ = pearsonr(xs, ys)
            layers.append(L); rs.append(r_val)
        ax.plot(layers, rs, marker=".", label=pos)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Layer"); ax.set_ylabel("Pearson r (probe vs system_followed)")
    ax.set_title(f"{MODEL_KEY}: per-layer correlation by extraction position")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_CSV.parent / "per_layer_correlation_by_position.png", dpi=120)
    plt.show()
else:
    print("No judge labels in rescored CSV — skipping per-layer correlation plot.")

In [ ]:
# Quick stats on the rescored CSV
import pandas as pd
df = pd.read_csv(OUT_CSV)
print("rescored rows:", len(df))
print("columns:", list(df.columns))
for pos in ("response_first", "response_mid", "response_last"):
    col = f"probe_score_{pos}_best_layer"
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        print(f"  {col:<40s} mean={s.mean():+.3f}  std={s.std():.3f}  finite={s.notna().sum()}")